In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

from sklearn.pipeline import Pipeline

In [2]:
train_df = pd.read_csv(
  "../data/processed/raw/train.csv"
)

validation_df = pd.read_csv(
  "../data/processed/raw/validation.csv"
)

In [3]:
print("Train shape: ", train_df.shape)
print("Validation shape: ", validation_df.shape)

train_df.head()

Train shape:  (18831, 7)
Validation shape:  (4010, 7)


,flags,instruction,category,intent,response,normalized_instruction,processed_instruction
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...,question about cancelling order {{order number}},question about cancelling order {{Order Number}}
1,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...,i need help cancelling puchase {{order number}},i need help cancelling puchase {{Order Number}}
2,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...,i need to cancel purchase {{order number}},I need to cancel purchase {{Order Number}}
3,BI,can you help me cancel order {{Order Number}}?,ORDER,cancel_order,"Of course, I'm here to assist you in canceling...",can you help me cancel order {{order number}}?,can you help me cancel order {{Order Number}}?
4,BCLN,"I can no longer afford order {{Order Number}},...",ORDER,cancel_order,I pick up what you're putting down that you're...,"i can no longer afford order {{order number}},...","I can no longer afford order {{Order Number}},..."


In [4]:
X_train = train_df['processed_instruction']
y_train = train_df["intent"]

X_val = validation_df['processed_instruction']
y_val = validation_df['intent']

print("Training samples: ", len(X_train))
print("Validation samples: ", len(X_val))

Training samples:  18831
Validation samples:  4010


### Base Pipeline

In [5]:
baseline_pipeline = Pipeline([
  (
    "tfidf",
    TfidfVectorizer(
      lowercase=True
    )
  ),
  (
    "classifier",
    LogisticRegression(
      max_iter=1000,
      random_state = 42
    )
  )
])

In [6]:
# Train

baseline_pipeline.fit(
  X_train,
  y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](27,)","['cancel_order','change_order','change_shipping_address',..., 'switch_account','track_order','track_refund']"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True


### Validation Prediction

In [7]:
y_val_pred = baseline_pipeline.predict(
  X_val
)

#### Compute Metrics

In [9]:
accuracy = accuracy_score(
  y_val,
  y_val_pred
)

macro_precision = precision_score(
  y_val,
  y_val_pred,
  average='macro'
)

macro_recall = recall_score(
  y_val,
  y_val_pred,
  average='macro'
)

macro_f1 = f1_score(
  y_val,
  y_val_pred,
  average='macro'
)

weighted_f1 = f1_score(
  y_val,
  y_val_pred,
  average="weighted"
)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Macro Precision:  {macro_precision:.4f}")
print(f"Macro Recall:   {macro_recall:.4f}")
print(f"Macro F1:   {macro_f1:.4f}")
print(f"Weighted F1:  {weighted_f1:.4f}")

Accuracy:  0.9955
Macro Precision:  0.9956
Macro Recall:   0.9956
Macro F1:   0.9956
Weighted F1:  0.9955


### Classification Report

In [10]:
print(
  classification_report(
    y_val,
    y_val_pred
  )
)

                          precision    recall  f1-score   support

            cancel_order       1.00      0.98      0.99       146
            change_order       0.97      0.99      0.98       151
 change_shipping_address       0.99      1.00      1.00       147
  check_cancellation_fee       1.00      1.00      1.00       142
           check_invoice       1.00      0.99      1.00       150
   check_payment_methods       1.00      1.00      1.00       150
     check_refund_policy       1.00      1.00      1.00       150
               complaint       0.99      1.00      1.00       149
contact_customer_service       1.00      0.99      1.00       149
     contact_human_agent       0.99      1.00      1.00       150
          create_account       1.00      0.97      0.99       160
          delete_account       0.98      1.00      0.99       149
        delivery_options       1.00      1.00      1.00       138
         delivery_period       1.00      1.00      1.00       149
         